# Topics around Paul — distribution over time, and differences across subsets

Exploration of the BERTopic model over the Paul concordance: **50 topics** fitted on **379,859
documents**, where a document is the ±10-token lemmatised window around one mention of Paul in
the JSTOR 2026-04 Philosophy ∪ Religion corpus (1900–2019).

The two questions this notebook is for:

1. **What is the relative distribution of topics over time?** (§7–8)
2. **How does that distribution — and its trajectory — differ across subsets?** (§9–12)

> **Read `TOPICS.md` at the repo root before quoting any topic number.** It holds the 13 decisions
> behind this model, the frozen parameters, and the limitations. This notebook only *reads* the
> fitted artifacts; it never fits anything (D11 — fitting belongs to `scripts/fit_paul_topics.py`).

## Three things to know before reading any number here

- **56.2% of the documents are in the outlier topic (`-1`).** HDBSCAN could not place them in any
  cluster. They are kept and reported, never reassigned (D9), so **"the topics" means the 43.8% the
  algorithm was willing to cluster** — not the corpus.
- **Documents are overlapping windows, not texts.** Adjacent hits in one article share tokens, so
  observations are **not independent**. No p-value, chi-square or confidence interval computed from
  them would be valid, and this notebook computes none. Distances are reported as descriptive
  magnitudes only.
- **Topics are term lists, not names.** Labels come from c-TF-IDF over lemmas, so they read like
  `jewish, jews, juif, judaism, jude, jüdisch`. That a single topic mixes languages is the intended
  behaviour (D6): it means the clustering is thematic rather than by language.

## Where the outline of a "standard" topic-model notebook does not apply

Sections 4–6 of a generic template (preprocess text → fit LDA/NMF → assign dominant topics) are
**already done, upstream and once**:

| generic step | here |
| --- | --- |
| preprocess text, build document-term matrix | lemmatisation happened in `prepare_jstor_app_data.py`; the model consumes `filtered_lemmata` |
| fit a topic model | `fit_paul_topics.py` produced the artifacts (§4.1 of `TOPICS.md`); the notebook must not re-fit |
| assign dominant topics | `docs.parquet` already carries a `topic` per document |

So §4–6 below **inspect the pre-fitted model** instead of rebuilding it. Everything after that
(the time distribution, the subset comparisons, the divergences) is what this notebook computes.

## 1. Setup

No modelling libraries: everything below calls `mops_app.topics_data`, the same streamlit-free
module the web app renders from. Reusing it rather than reimplementing means the notebook and the
`/mops/` page cannot silently disagree — and `scripts/check_mops_app.py` pins that they don't.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# The notebook runs from scripts/, and `mops_app` lives beside it. The systemd unit
# and the app set this up for themselves; a bare kernel does not.
sys.path.insert(0, str(Path.cwd()))

from mops_app import data                       # bin_edges / bin_label / DEFAULT_BIN_LENGTH
from mops_app import topics_data as td          # the topic-model layer (no streamlit)

# Keep the editor responsive: `docs` is 379,859 x 18 and the subset tables are
# ~50 rows x 8 columns, so a wide frame must wrap rather than print unbounded.
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.show_dimensions", True)

# One bin length for the whole notebook; 20 = bidecades, the project's convention.
BIN_LENGTH = data.DEFAULT_BIN_LENGTH

# The topic the outlier rows carry. Named rather than inlined so `!= -1` never
# becomes `!= 0` by accident.
OUTLIER = td.OUTLIER_TOPIC

print(f"mops_app.topics_data | bin length {BIN_LENGTH} | outlier topic {OUTLIER}")
print("runs on disk:", td.available_runs())

## 2. Load and inspect the fitted artifacts

There is no CSV to parse and no `pd.to_datetime` to do: the corpus already carries an integer
`year` and the model's artifacts are parquet. Three files matter, and the distinction between them
is the reason this notebook cannot just read a precomputed table:

| artifact | what it is | used for |
| --- | --- | --- |
| `docs.parquet` | one row per fitted document: `topic`, 2-D map position, and the filterable metadata | **everything** — the user's filters must be applied *before* counting |
| `topics_info.parquet` | the topic table: size, c-TF-IDF terms | labelling |
| `report.json` | the run's own record of parameters, counts and versions | provenance |

The precomputed `topics_over_time.parquet` / `topics_per_class.parquet` describe the **unfiltered**
fit, so this notebook recomputes from `docs.parquet` instead. `check_mops_app.py` asserts that with
no filters the two agree — the recomputation is verified, not trusted.

In [ ]:
# `full` is the production fit. A sample run is available via td.load_docs(run="sample100k")
# but it is not the published model.
RUN = td.TOPIC_RUN

docs = td.load_docs(run=RUN)          # per-document topic + map coords + metadata
info = td.load_topics_info(run=RUN)   # topic sizes and c-TF-IDF terms
report = td.load_report(run=RUN)      # the run's own record

t, d = report["topics"], report["documents"]
print(f"run '{report['slug']}' fitted {report['started']}")
print(f"  documents      {d['fitted']:,}  ({d['articles']:,} articles, "
      f"{d['journals']} journals, {d['year_min']}–{d['year_max']})")
print(f"  topics         {t['n_topics']} (+ outlier)")
print(f"  outlier share  {t['outlier_share']:.1%}  ({t['outlier_topic_docs']:,} documents)")
print(f"  min_cluster_size {t['min_topic_size']} = {t['min_topic_size'] / d['fitted']:.3%} of docs"
      f" | min_samples {t['min_samples']} | epsilon {t['cluster_selection_epsilon']}")
print(f"  encoder        {report['decisions']['D6_embedding_model']}")
print(f"  versions       bertopic {report['versions']['bertopic']}, "
      f"sentence-transformers {report['versions']['sentence-transformers']}, "
      f"numpy {report['versions']['numpy']}, pandas {report['versions']['pandas']}")

# This must equal the outlier share in docs.parquet; if it ever does not, the notebook
# is reading artifacts from different runs.
assert abs((docs["topic"] == OUTLIER).mean() - t["outlier_share"]) < 5e-5, "docs/report disagree"
print("\ndocs and report agree ✓")

In [ ]:
# What is actually in the table, and which columns can define a subset.
print("columns:", list(docs.columns))
print()
print(docs.dtypes.to_string())
print()
docs.head(3)

In [ ]:
# Candidate subset-defining columns, with their value counts. `None` is a real
# category for the journal annotations: ~33% of documents belong to journals the
# annotation sheet does not cover, so a discipline filter also narrows the corpus.
for col in ["tier", "language", "journal-discipline", "journal-confession"]:
    vc = docs[col].value_counts(dropna=False)
    share = (vc / len(docs)).round(3)
    print(f"--- {col} ---")
    print(pd.DataFrame({"n": vc, "share": share}).to_string())
    print()

## 3. Time periods

The `year` column is already an integer, so there is nothing to parse — and `dt.to_period()` would
be the wrong tool anyway: the project's unit of time is the **bidecade**, and the binning lives in
`mops_app.data.bin_edges` / `bin_label` so that the notebook, the app and every published figure
cut the century at the same places. `docs.parquet` even carries a precomputed `bidecade` column;
we recompute the bins from `year` only because that keeps `bin_length` a knob.

In [ ]:
YEAR_RANGE = (int(docs["year"].min()), int(docs["year"].max()))
edges = data.bin_edges(*YEAR_RANGE, BIN_LENGTH)
labels = [data.bin_label(e) for e in edges]

print(f"year range {YEAR_RANGE[0]}–{YEAR_RANGE[1]} | {len(edges)} bins of {BIN_LENGTH}y")
print(labels)

# Documents per bin. The corpus grows by roughly an order of magnitude across the
# century, which is precisely why every number below is a *share*: a rising raw
# count would mostly measure that growth (TOPICS.md D12).
per_bin = (docs.assign(**{"bin": pd.cut(docs["year"], bins=[e[0] for e in edges] + [edges[-1][1] + 1],
                                        right=False, labels=labels)})
           .groupby("bin", observed=True).size().rename("documents"))
print()
print(per_bin.to_frame().assign(share=(per_bin / len(docs)).round(4)).to_string())

## 4–6. The document, the model and the assignment (inspected, not built)

Normally these sections preprocess text, fit a model and assign topics. All three are finished
upstream and frozen — re-fitting here would produce a *different* model than the one the app, the
record and any published figure refer to. What is useful instead is to see what a document is, and
what the topics came out as.

In [ ]:
# A topic is a cluster of ±10-token windows, so its c-TF-IDF terms are what recurs
# in that neighbourhood. Labels are term lists rather than names: naming is left to
# `topic_names.json` / a future curation step (TOPICS.md §8), and no analysis below
# depends on a name.
size_table = td.topic_sizes(docs, info)
print(f"{len(size_table)} topics present ({size_table['Topic'].ne(OUTLIER).sum()} + outlier)")
print()
top = size_table.head(26).copy()
print(f"{'topic':>7} {'docs':>8} {'share':>7} {'of clustered':>13}  terms")
for r in top.itertuples():
    tag = "OUTLIER" if r.Topic == OUTLIER else f"t{r.Topic}"
    print(f"{tag:>7} {r.n_docs:>8,} {r.share:>7.3%} {r.share_modelled:>13.3%}  "
          f"{str(r.top_terms_str)[:64]}")

### The outlier share is the first thing to look at, and it is not constant

Before comparing anything, note that `-1` is not leftover noise to discard. It is the largest single
group in the corpus, and its size **differs between the subsets** — that makes it a confound as
well as a finding. `share_modelled` divides by the *clustered* documents, so a subset with a bigger
unclustered remainder has all of its other topic shares inflated by that normalisation alone.

In [ ]:
# The single number that constrains every reading below.
n_modelled = int((docs["topic"] != OUTLIER).sum())
print(f"clustered (modelled) documents : {n_modelled:>9,}  {n_modelled / len(docs):>7.1%}")
print(f"outlier topic (-1)            : {len(docs) - n_modelled:>9,}  "
      f"{1 - n_modelled / len(docs):>7.1%}")
print()
print("So 'the topics' below describes the clustered part of the corpus. A share of")
print("0.10 in a bin means '10% of that bin's documents', not '10% of its discourse'.")

## 7. Relative topic distribution per time period

`td.topic_trends` returns a bins × topics matrix, normalised **within each bin** so the columns are
comparable despite the corpus growing ~8×. Two knobs:

- `BIN_LENGTH` — 20 (bidecades, the default) gives 6 bins; 10 gives decades and 12 bins, at the cost
  of noisier early bins;
- `metric` — `share` (of all documents in the bin, outlier topic included) or `share_modelled`
  (of the clustered ones only). They answer different questions and **can move in opposite
  directions** — see the demonstration two cells down.

In [ ]:
trends_all = td.topic_trends(docs, bin_length=BIN_LENGTH, year_range=YEAR_RANGE, metric="share")

# Column sums are exactly 1 (every document is in exactly one topic, including the
# outlier topic), which is the check that the normalisation is the intended one.
print("shape:", trends_all.shape, "| bin sums:", [round(float(v), 6) for v in trends_all.sum(axis=1)])

# A readable slice: the largest topics, whose share is big enough to follow.
largest = [t for t in size_table["Topic"].tolist() if t != OUTLIER][:9]
show = trends_all[largest].copy()
show.columns = [f"t{c}" for c in show.columns]
print()
print(show.round(4).to_string())

## 8. Topic share over time

Two views of the same matrix: a **stacked area** shows the composition of each period, and a
**line chart** makes individual trajectories readable. Both label topics by number *and* terms,
because a bare `t37` is unreadable and a bare term list is ambiguous across topics.

In [ ]:
def labelled(matrix: pd.DataFrame, n: int, *, include_outlier: bool = False) -> pd.DataFrame:
    """Take the n largest columns of a bins x topics matrix, with readable names.

    Returns a frame indexed by bin with columns like `t12 — jewish, jews, juif`.
    Names are truncated: the legend is not the place to read a full term list.
    """
    sizes = docs["topic"].value_counts()
    cols = [t for t in sizes.index if include_outlier or t != OUTLIER][:n]
    cols = [c for c in cols if c in matrix.columns]
    out = matrix[cols].copy()
    out.columns = [td.topic_label(int(c), info, width=34) for c in cols]
    return out


def long_form(matrix: pd.DataFrame, **extra) -> pd.DataFrame:
    """bins x topics -> long frame for plotly, preserving bin order."""
    order = {lab: i for i, lab in enumerate(matrix.index)}
    out = (matrix.rename_axis("bin").reset_index()
           .melt(id_vars="bin", var_name="topic", value_name="value"))
    out["_order"] = out["bin"].map(order)
    return out.sort_values("_order").drop(columns="_order").reset_index(drop=True)


# --- stacked composition -------------------------------------------------------
area = long_form(labelled(trends_all, 12))
fig = px.area(area, x="bin", y="value", color="topic",
              labels={"bin": "", "value": td.TOPIC_METRICS["share"], "topic": ""},
              title="Composition of each period: the 12 largest topics")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.5))
fig.show()

In [ ]:
# --- individual trajectories ---------------------------------------------------
fig = px.line(long_form(labelled(trends_all, 10)), x="bin", y="value", color="topic",
              markers=True,
              labels={"bin": "", "value": td.TOPIC_METRICS["share"], "topic": ""},
              title="Trajectory of the 10 largest topics")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.5), height=520)
fig.show()

# Read this against the outlier topic, which is in a different league in size and is
# the denominator of nothing here: including it keeps the shares summing to 1.
fig = px.line(long_form(trends_all[[OUTLIER]].rename(columns={OUTLIER: "outlier (−1)"})),
              x="bin", y="value", markers=True,
              labels={"bin": "", "value": td.TOPIC_METRICS["share"], "topic": ""},
              title="The outlier share, for reference (it moves, so it is a confound)")
fig.update_layout(showlegend=False, height=320)
fig.show()

In [ ]:
# The two metrics are not interchangeable, and the difference is not cosmetic.
# `share_modelled` removes the outlier topic from the denominator, so it *rises*
# whenever the outlier share falls -- mechanically, with nothing changing in what is
# discussed. A topic can therefore appear to grow under one metric and shrink under
# the other. This cell shows the same topic under both.
TARGET = int(size_table.loc[size_table["Topic"] != OUTLIER, "Topic"].iloc[0])

modelled = td.topic_trends(docs, bin_length=BIN_LENGTH, year_range=YEAR_RANGE,
                           metric="share_modelled")
compare = pd.DataFrame({
    "share": trends_all[TARGET],
    "share_modelled": modelled[TARGET],
    "outlier_share (1 - share of -1)": 1 - trends_all[OUTLIER],
})
compare.index.name = "bin"
print(f"topic {td.topic_label(TARGET, info, width=70)}")
print()
print(compare.round(4).to_string())

fig = px.line(long_form(compare), x="bin", y="value", color="topic", markers=True,
              labels={"bin": "", "value": "share", "topic": ""},
              title=f"t{TARGET}: the same topic under two normalisations")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.4), height=420)
fig.show()

## 9. Define the subsets

One dict, reused by every cell below, so each comparison is over exactly the same subsets.

Two notes on the design:

- **The subsets overlap, and that is intentional.** A tier-A English document belongs to two
  subsets. Subsets are *lenses on the corpus*, not a partition, so the shares inside a subset must
  be read against the `overall` column rather than added up. For the same reason a single
  "subset membership" column is the wrong shape here — a document has several memberships.
- **The tier is a scan property, not a topic property.** Tier A = the text explicitly identifies
  the apostle ("apostle Paul", "Paul of Tarsus"), B = strong ("Pauline", "Paulus"), C = bare
  "Paul" resolved by context, which is the largest and by far the noisiest (the corpus is full of
  people called Paul). Comparing tiers is therefore also comparing *detection precision*.

In [ ]:
# `td.filter_docs` takes the same keyword arguments as the app's filters. `overall`
# must come first: it is the reference column of every comparison below.
SUBSETS = {
    "overall": docs,
    "tier A": td.filter_docs(docs, tiers=["A"]),
    "tier B": td.filter_docs(docs, tiers=["B"]),
    "tier C": td.filter_docs(docs, tiers=["C"]),
    # `en` and `eng` are both English in this corpus and must travel together.
    "English": td.filter_docs(docs, languages=["eng", "en"]),
    "French": td.filter_docs(docs, languages=["fre"]),
    "German": td.filter_docs(docs, languages=["ger"]),
    "biblical/early Christian": td.filter_docs(docs, disciplines=["biblical/early Christian"]),
    "theology": td.filter_docs(docs, disciplines=["theology"]),
    "philosophy/ethics": td.filter_docs(docs, disciplines=["philosophy/ethics"]),
    "Catholic": td.filter_docs(docs, confessions=["Catholic"]),
    "Protestant": td.filter_docs(docs, confessions=["Protestant"]),
}

# Validate before comparing: sizes, year coverage, and — crucially — the outlier
# share, because it is the denominator of `share_modelled` and therefore a confound
# for any subset difference (see the note printed with the table).
coverage = pd.DataFrame({
    label: {"n_docs": len(df),
            "year_min": int(df["year"].min()) if len(df) else None,
            "year_max": int(df["year"].max()) if len(df) else None,
            "journals": int(df["journal"].nunique()) if len(df) else 0}
    for label, df in SUBSETS.items()
}).T
print(coverage.to_string())

In [ ]:
# The confound, in one table. A subset with a *larger* unclustered remainder has its
# remaining topic shares inflated by the `share_modelled` normalisation alone, so two
# subsets can differ in every topic share without differing in what is discussed.
out_shares = td.outlier_shares(SUBSETS)
print(out_shares.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print(f"spread across subsets: {out_shares['outlier_share'].min():.1%}"
      f" – {out_shares['outlier_share'].max():.1%}"
      f"  (range {out_shares['outlier_share'].max() - out_shares['outlier_share'].min():.1%})")

## 10. Compare the relative topic distribution across subsets

`td.subset_shares` normalises **within each subset**, so the columns are comparable even though the
subsets differ hugely in size (4,652 for philosophy/ethics against 379,859 overall). `metric="share"`
is the default here on purpose: it includes the outlier topic in the denominator, so the ~4pp spread
in outlier share shown above is carried in the numbers rather than hidden behind them.

Then `td.diff_from_reference` subtracts the `overall` column: **positive = the topic occupies a
larger share of this subset than of the corpus as a whole.**

The differences must be read against the base rate, so the raw shares are kept on screen beside
them: +1.5pp on a topic that is 3% of the corpus is a different claim from +1.5pp on a topic that
is 20% of it.

In [ ]:
SUBSET_METRIC = "share"   # try "share_modelled" to see the alternative question

shares = td.subset_shares(SUBSETS, metric=SUBSET_METRIC)
print(f"topics x subsets: {shares.shape} | column sums "
      f"{[round(float(v), 4) for v in shares.sum(axis=0)]}")
print()
print(shares.head(10).round(4).to_string())

deltas = td.diff_from_reference(shares, "overall")
print()
print(f"differences vs overall ({shares.shape[1] - 1} subsets), pp for the raw shares")

In [ ]:
# Which topics discriminate most between subsets? Rank by the largest absolute
# difference any subset shows, and print the topic's overall size next to it so a
# large gap on a rare topic is not mistaken for a large effect.
TOP_N = 20
rank = deltas.abs().max(axis=1).rename("max_abs_delta")
size_by_topic = docs["topic"].value_counts().rename("overall_n")
topics_ranked = (rank.to_frame()
                 .join(shares["overall"].rename("overall_share"))
                 .join(size_by_topic)
                 .sort_values("max_abs_delta", ascending=False))

print(f"{'topic':>7} {'worst gap':>10} {'overall':>9} {'docs':>9}  terms")
for t, r in topics_ranked.head(TOP_N).iterrows():
    tag = "OUTLIER" if t == OUTLIER else f"t{t}"
    terms = info.loc[info["Topic"] == t, "top_terms_str"]
    print(f"{tag:>7} {r.max_abs_delta:>10.4f} {r.overall_share:>9.4f} "
          f"{int(r.overall_n):>9,}  {str(terms.squeeze() if len(terms) else '')[:56]}")

In [ ]:
# Heatmap of the differences, restricted to the topics that discriminate most.
# A diverging scale centred on 0: red = over-represented relative to the corpus,
# blue = under-represented. The outlier row is kept visible, because its variation
# is the confound the other rows are read against.
sel = [t for t in topics_ranked.head(TOP_N).index]
heat = deltas.loc[sel].copy()
heat.index = [("OUTLIER" if t == OUTLIER else f"t{t}") for t in heat.index]

fig = px.imshow(heat, aspect="auto", color_continuous_scale="RdBu_r",
                color_continuous_midpoint=0.0, origin="upper",
                labels=dict(x="subset", y="topic", color="difference"),
                title=f"Topic share minus the corpus average (metric: {SUBSET_METRIC})")
fig.update_layout(height=640, coloraxis_colorbar_title="Δ share")
fig.show()

## 11. How far apart are the subsets? Two complementary measures

The per-topic differences above say *where* subsets differ. These two say *how much* they differ
overall, and over time:

1. **`td.distribution_distance`** — one number per subset: the distance between its whole topic
   vector and the `overall` vector. Three options: Jensen–Shannon (`js`, bounded 0–1, 0 = identical),
   total variation (`tv`, the share of mass that must move), cosine (`cosine`).
2. **`td.distance_over_time`** — the same distance computed **per bin**, which answers whether the
   subsets have grown more or less alike, something a single overall number cannot show.

> **No chi-square test here, deliberately.** A χ² test assumes independent observations. Our
> documents are overlapping ±10-token windows — adjacent hits in one article share tokens — so the
> assumption fails and any p-value would be an artefact of that, not evidence. These are reported as
> descriptive magnitudes on a fixed scale; they rank subsets, they do not test them. This is
> `TOPICS.md` §7 limitation 1, made operational rather than acknowledged.

In [ ]:
# All three measures side by side. They must agree on the *ordering* (that is the
# only thing being claimed); the magnitudes are not comparable across rows of
# different measures, only within one.
for method in ("js", "tv", "cosine"):
    dd = td.distribution_distance(shares, "overall", method=method)
    line = "  ".join(f"{r.subset} {r.distance:.3f}" for r in dd.itertuples())
    print(f"{method:>7}: {line}")

print()
print(f"js  — {td.DISTANCES['js']}")
print(f"tv  — {td.DISTANCES['tv']}")
print(f"cos — {td.DISTANCES['cosine']}")
print()
print("Note: cosine distances are tiny and poorly separated here. Topic vectors are")
print("non-negative and share a large common component, so the cosine is dominated by")
print("that shared mass. js and tv are the informative ones for shares.")

In [ ]:
# All subsets at once, as a long frame: (subset, bin, topic, value). This is what
# feeds both the per-topic facet plots and the per-bin distances below. Long rather
# than a dict of matrices precisely because the comparison is the point.
#
# `min_docs_per_bin` is a noise floor defined here so every later cell uses the same
# one. 250 is roughly a quarter of the smallest subset's typical bin.
MIN_DOCS_PER_BIN = 250

all_trends = td.trends_by_subset(SUBSETS, bin_length=BIN_LENGTH,
                                year_range=YEAR_RANGE, metric=SUBSET_METRIC)
print(f"trends: {all_trends.shape[0]:,} rows | "
      f"{all_trends['subset'].nunique()} subsets | "
      f"{all_trends['topic'].nunique()} topics | {all_trends['bin'].nunique()} bins")
print("subsets:", list(dict.fromkeys(all_trends["subset"])))

In [ ]:
# The distance per bin. `overall` is the reference, so it is excluded from the output.
#
# `min_docs` masks bins where a subset has too few documents for its share vector to
# be stable. Without a floor, a subset's earliest sparse bins produce jumpy shares
# that dominate the picture. Masks are returned as NaN, so a plot shows a gap rather
# than silently bridging it. `docs_per_bin` comes from the `count` metric.
counts_by_bin = td.trends_by_subset(SUBSETS, bin_length=BIN_LENGTH,
                                    year_range=YEAR_RANGE, metric="count")
docs_per_bin = counts_by_bin.groupby(["subset", "bin"])["value"].sum()

dist_time = td.distance_over_time(all_trends, "overall", method="js",
                                 min_docs=MIN_DOCS_PER_BIN, docs_per_bin=docs_per_bin)
wide_dist = dist_time.pivot(index="bin", columns="subset", values="distance")
wide_dist = wide_dist.reindex([lab for lab in labels if lab in wide_dist.index])
print(f"masked cells (subset x bin below {MIN_DOCS_PER_BIN} documents): "
      f"{int(wide_dist.isna().sum().sum())} of {wide_dist.size}")
print()
print(wide_dist.round(4).to_string())

fig = px.line(long_form(wide_dist), x="bin", y="value", color="topic", markers=True,
              labels={"bin": "", "value": "Jensen–Shannon distance from the corpus",
                      "topic": "subset"},
              title="Have the subsets grown more or less like the corpus overall?")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.5), height=460)
fig.show()

## 12. Subset trajectories side by side

The figures above compare subsets *on average*. This section asks the interaction question: **do the
subsets' trajectories differ?** That is genuinely different information. A topic can have the same
average share in two subsets while rising in one and falling in the other, and the average-only view
would call those identical.

Once a topic is chosen, one panel per topic with one line per subset makes the comparison direct.
Topics are labelled by number and terms; the `overall` line is included as a reference, because a
trajectory that merely follows the corpus is not a finding about the subset.

In [ ]:
# Which topics do the subsets disagree about *over time*?
#
# `td.trajectory_gap` measures, for each pair of subsets, how far apart their share
# series are for one topic (mean absolute difference across bins). We take each
# topic's worst pair as its "contested" score and rank by it.
#
# REMINDER: this is a descriptive magnitude, not a test. The documents are
# overlapping windows and non-independent, so no p-value belongs here.
rows = []
for t in size_table.loc[size_table["Topic"] != OUTLIER, "Topic"].head(30):
    gap = td.trajectory_gap(all_trends, int(t))
    if gap.empty:
        continue
    worst = gap.iloc[0]
    rows.append({"topic": int(t), "worst_pair": f"{worst.subset_a} vs {worst.subset_b}",
                 "mean_abs_gap": worst.mean_abs_gap, "max_abs_gap": worst.max_abs_gap,
                 "overall_share": float(shares.loc[t, "overall"]) if t in shares.index else np.nan})

contested = pd.DataFrame(rows).sort_values("mean_abs_gap", ascending=False).reset_index(drop=True)
contested["terms"] = contested["topic"].map(
    lambda t: str(info.loc[info["Topic"] == t, "top_terms_str"].squeeze())[:52])
print(contested.head(12).to_string(index=False, float_format=lambda x: f"{x:.5f}"))

In [ ]:
# Facet: one panel per topic, one line per subset. Rows are a fixed order across
# panels so the subsets stay comparable within a panel *and* between panels.
FACET_TOPICS = contested["topic"].head(6).tolist()

facet = all_trends[all_trends["topic"].isin(FACET_TOPICS)].copy()
facet["panel"] = facet["topic"].map(lambda t: td.topic_label(int(t), info, width=42))
facet = facet.sort_values(["panel", "subset", "bin"])

# Drop the earliest bins where a subset is below the noise floor, for the same
# reason `distance_over_time` masks them: a share from 12 documents is not a share.
facet = facet.merge(docs_per_bin.rename("n_docs").reset_index(),
                    left_on=["subset", "bin"], right_on=["subset", "bin"], how="left")
facet.loc[facet["n_docs"] < MIN_DOCS_PER_BIN, "value"] = np.nan

fig = px.line(facet, x="bin", y="value", color="subset", facet_col="panel",
              facet_col_wrap=3, markers=True,
              labels={"bin": "", "value": f"share ({SUBSET_METRIC})", "subset": ""},
              title="Trajectories of the most contested topics: one panel per topic, "
                    "one line per subset")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.28), height=620)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
# One topic's full subset profile, and the pair-level detail behind its ranking.
# This is the drill-down from the ranking above: the table says *which* pair
# disagrees most, the chart says *how* the topic sits in each subset overall.
PICK = int(contested.loc[0, "topic"])
gap_detail = td.trajectory_gap(all_trends, PICK)
print(f"{td.topic_label(PICK, info, width=80)}")
print(f"worst pair: {contested.loc[0, 'worst_pair']}\n")
print(gap_detail.head(8).to_string(index=False, float_format=lambda x: f"{x:.5f}"))

profile = (shares.loc[PICK].rename("share").to_frame()
           .join(td.outlier_shares(SUBSETS).set_index("subset")["outlier_share"])
           .sort_values("share", ascending=False))
fig = px.bar(profile.reset_index().rename(columns={"index": "subset"}),
             x="subset", y="share", title=f"t{PICK}: share in each subset (metric {SUBSET_METRIC})",
             labels={"share": f"share ({SUBSET_METRIC})", "subset": ""})
fig.update_layout(height=400)
fig.show()

## Caveats — what these numbers may not be used to claim

Kept here rather than in `TOPICS.md` alone, because a notebook is where numbers get copied out of.

1. **More than half the corpus is unclustered.** 56.2% of documents are the outlier topic, so
   "the topics" means the clustered 43.8%. Any sentence of the form "most of the corpus is about X"
   is unsupportable from this model.
2. **The documents are not independent.** They are overlapping ±10-token windows, and adjacent hits
   in one article share tokens. This is why this notebook reports **no p-values, chi-square tests or
   confidence intervals**: the independence assumption fails, so such a figure would be an artefact
   of the windowing, not evidence. Distances are ordinals on a fixed scale.
3. **The outlier share differs between subsets** (62% for tier A down to 53% for tier C) and that is
   a **confound**, not just a finding: it is the denominator of `share_modelled`, so a subset with a
   bigger unclustered remainder has its other topics inflated by the normalisation alone. Prefer
   `metric="share"` when comparing subsets, and report the outlier share alongside any claim.
4. **Topics are descriptive, not causal or authorial.** They describe recurring lexical-semantic
   configurations near the name, not arguments, intentions or intellectual traditions. A topic that
   grows is a pattern of vocabulary that grows; the explanation for it is not in the model.
5. **The model is validated by inspection, not by labels.** There is no gold-standard topic
   annotation for this corpus. Quality rests on term readability and the checks in
   `check_mops_app.py`.
6. **Latin and the smaller language tails are excluded** (D2/D13) — French `fre`, not `fr`, was a
   real bug that emptied 66,059 documents until D13 caught it. The model describes the English-,
   French- and German-language literature of this field only.
7. **Tier C is noisy.** Bare `Paul` cannot be told from the many other people called Paul by ±46
   characters of context, so tier-C "topics" partly describe detection noise. Tier A and B are the
   trustworthy counts.
8. **A topic's label is not its meaning.** The terms are what the cluster's c-TF-IDF ranks highest;
   reading a topic means reading its KWIC lines, which the app's **Concordance browser** page or
   `data.search_concordance` will show for the same filter set.

## Reproducing and re-running

This notebook reads artifacts; it never fits. Fitting belongs to the script, so that a number here
and a number on the website always come from the same run (D11).

In [ ]:
print(f"""artifacts : {td.TOPICS_DIR}
run       : {RUN}  (report.json holds the resolved parameters)
encoder   : {report['decisions']['D6_embedding_model']}

re-fit / re-check (from scripts/, using the topic stack in requirements-topics.txt):

    ../mops_venv/bin/python fit_paul_topics.py --min-topic-size-frac 0.002   # ~11 min
    ../mops_venv/bin/python fit_paul_topics.py --verify
    ../mops_venv/bin/python check_mops_app.py                               # app + parity

the three stages that dominate a fit are all cached, so re-clustering costs ~20 s:

    embeddings.npy   {td.paths_for(RUN)['embeddings'].stat().st_size / 1e6:>7,.0f} MB   (D6/D7, ~5 min on GPU)
    reduced5.npy     {td.paths_for(RUN)['reduced5'].stat().st_size / 1e6:>7,.0f} MB   (5-d clustering space, ~5 min)
    xy.npy           {td.paths_for(RUN)['xy'].stat().st_size / 1e6:>7,.0f} MB   (2-d app map, ~15 min — the slowest stage)

before touching any HDBSCAN parameter, read TOPICS.md D8.2: an epsilon tuned on a
100k sample merged 52.6% of the corpus into one topic at full scale while *improving*
the outlier metric. Sweep at the scale you intend to fit.""")